In [30]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler

# df = pd.read_csv("../train/cscada_labeled.csv") 

# # print(df.columns)

# feature_cols = [
#     'packet_count', 'total_bytes',
#        'mean_packet_size', 'std_packet_size', 'iat_mean', 'iat_std',
#        'unique_func_codes', 'exception_count'
# ]

# X = df[feature_cols].fillna(0) #to remove NaNs
# y = df["is_attack"]

# # Optional: scale features
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_scaled, y, test_size=0.2, random_state=42, stratify=y
# )

# print("Train size:", X_train.shape)
# print("Test size:", X_test.shape)

# #we now split the data into training data and test data

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../train/cscada_labeled.csv")

feature_cols = [
    'packet_count', 'total_bytes',
    'mean_packet_size', 'std_packet_size',
    'iat_mean', 'iat_std',
    'unique_func_codes', 'exception_count'
]

X = df[feature_cols].fillna(0)
y = df["is_attack"]

# First split: 80% train, 20% temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Second split: 10% val, 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# Scale AFTER splitting (important to avoid data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)




Train: (648886, 8)
Validation: (81111, 8)
Test: (81111, 8)


In [31]:
#supervised models: Logistic regression (lightweight, apparently)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Logistic Regression 1:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression 1:
              precision    recall  f1-score   support

           0       0.82      1.00      0.90     66469
           1       0.52      0.00      0.01     14642

    accuracy                           0.82     81111
   macro avg       0.67      0.50      0.45     81111
weighted avg       0.77      0.82      0.74     81111



In [32]:
#supervised models: Logistic regression (lightweight, apparently)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Logistic Regression 2:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression 2:
              precision    recall  f1-score   support

           0       1.00      0.47      0.64     66469
           1       0.29      1.00      0.45     14642

    accuracy                           0.57     81111
   macro avg       0.65      0.74      0.55     81111
weighted avg       0.87      0.57      0.61     81111



In [33]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest 1:")
print(classification_report(y_test, y_pred_rf))


Random Forest 1:
              precision    recall  f1-score   support

           0       0.90      0.99      0.94     66469
           1       0.94      0.50      0.65     14642

    accuracy                           0.90     81111
   macro avg       0.92      0.75      0.80     81111
weighted avg       0.91      0.90      0.89     81111



In [34]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest 2:")
print(classification_report(y_test, y_pred_rf))


Random Forest 2:
              precision    recall  f1-score   support

           0       1.00      0.76      0.86     66469
           1       0.48      1.00      0.64     14642

    accuracy                           0.80     81111
   macro avg       0.74      0.88      0.75     81111
weighted avg       0.91      0.80      0.82     81111



In [35]:
print("Random Forest attempt 3:")
y_probs = rf.predict_proba(X_test)[:, 1]

for thresh in [0.5, 0.6, 0.7, 0.8]:
    y_pred = (y_probs > thresh).astype(int)
    print(f"\nThreshold: {thresh}")
    print(classification_report(y_test, y_pred))


Random Forest attempt 3:

Threshold: 0.5
              precision    recall  f1-score   support

           0       1.00      0.76      0.86     66469
           1       0.48      1.00      0.64     14642

    accuracy                           0.80     81111
   macro avg       0.74      0.88      0.75     81111
weighted avg       0.91      0.80      0.82     81111


Threshold: 0.6
              precision    recall  f1-score   support

           0       1.00      0.77      0.87     66469
           1       0.49      0.99      0.65     14642

    accuracy                           0.81     81111
   macro avg       0.74      0.88      0.76     81111
weighted avg       0.90      0.81      0.83     81111


Threshold: 0.7
              precision    recall  f1-score   support

           0       0.91      0.99      0.95     66469
           1       0.92      0.53      0.67     14642

    accuracy                           0.91     81111
   macro avg       0.91      0.76      0.81     81111
w

In [36]:
#isolation forest, but no benign dataset used
from sklearn.ensemble import IsolationForest

X_train_benign = X_train[y_train == 0]

iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X_train_benign)

y_pred_iso = iso.predict(X_test)

# Convert:
# -1 = anomaly → attack (1)
# 1 = normal → benign (0)
y_pred_iso = [1 if x == -1 else 0 for x in y_pred_iso]

print("Isolation Forest 1:")
print(classification_report(y_test, y_pred_iso))


Isolation Forest 1:
              precision    recall  f1-score   support

           0       0.82      0.82      0.82     66469
           1       0.19      0.19      0.19     14642

    accuracy                           0.71     81111
   macro avg       0.50      0.50      0.50     81111
weighted avg       0.71      0.71      0.71     81111



In [37]:
#this is trained on benign data and then tested on attack
benign_df = pd.read_csv("../train/benign_flows.csv")

X_benign = benign_df[feature_cols].fillna(0)
X_benign_scaled = scaler.fit_transform(X_benign)

iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X_benign_scaled)

comp_df = pd.read_csv("../train/cscada_labeled.csv")

X_comp = comp_df[feature_cols].fillna(0)
X_comp_scaled = scaler.transform(X_comp)

y_true = comp_df["is_attack"]

y_pred = iso.predict(X_comp_scaled)
y_pred = [1 if x == -1 else 0 for x in y_pred]

print("Isolation forest 2:")
print(classification_report(y_true, y_pred))



Isolation forest 2:
              precision    recall  f1-score   support

           0       0.84      0.72      0.78    664681
           1       0.23      0.38      0.29    146427

    accuracy                           0.66    811108
   macro avg       0.54      0.55      0.53    811108
weighted avg       0.73      0.66      0.69    811108

